# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

*Citation:* Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their respective fields using their `@id`s.

In [ ]:
# Discover all available record sets (`cr:RecordSet`) and their structure
record_sets = metadata.record_sets  # This is a list of RecordSet objects
print(f"Found {len(record_sets)} record sets.\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for f in rs.fields:
            print(f"    - @id: {f.id}\n      name: {f.name}\n      dataType: {f.data_type}")
    print('-' * 60)

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. Record sets and fields are referenced by their `@id`.

In [ ]:
# Extract data from all record sets by @id
dfs = {}
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"All record set @ids: {record_set_ids}\n")

for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"  DataFrame shape: {df.shape}")
    dfs[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}\n")

# Pick the first record set for demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Columns in record set '{example_record_set_id}': {dfs[example_record_set_id].columns.tolist()}")
    dfs[example_record_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Below we demonstrate: filtering records for a numeric field, normalizing the field, and grouping by a categorical field. All fields are referenced by their `@id`.

In [ ]:
# ---
# You may need to adjust these field @ids depending on the RecordSet's schema

if record_set_ids:
    # Inspect the first record set for common mass spec numeric fields
    df = dfs[example_record_set_id]
    print(f"DataFrame columns: {df.columns.tolist()}")
    # Try to find a likely numeric field. If you know a specific field @id, set it here.
    # Example choices: 'Coverage_percent', 'Peptide_count', 'MW', 'pI'
    # Use the exact @id, which should appear as a DataFrame column
    # For demo, pick the first numerical column we find
    numeric_field_id = None
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Try to coerce a likely numeric field (exclude the first column if it's an id)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if df[col].dtype in ['float64', 'int64']:
                    numeric_field_id = col
                    break
            except Exception:
                continue
    
    if numeric_field_id is not None:
        print(f"Numeric field (by @id): {numeric_field_id}")
        # Set a threshold for demonstration
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field (e.g. sample type, condition)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                values = df[col].nunique()
                if 1 < values < 30:  # Only group by columns with reasonable number of groups
                    group_field_id = col
                    break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found in this RecordSet.")
else:
    print("No record sets loaded to perform EDA.")

## 5. Visualization
Visualize a numeric field distribution and its normalized version. All fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id is not None:
    # Histogram of the field
    plt.figure(figsize=(12,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of field '@id'={numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Normalized field (if present in filtered_df)
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(12,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=40, kde=True)
        plt.title(f"Normalized distribution of field '@id'={numeric_field_id}")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel('Count')
        plt.show()
else:
    print("No numeric field available to visualize.")

## 6. Conclusion
This notebook loaded the FAIR^2 dataset via the Croissant schema and explored record set structures, extracted the main tabular data by referencing record sets and fields with their `@id`, and demonstrated basic filtering and visualization using those IDs for reproducibility and code clarity.

- The dataset provides structured records of proteins identified by mass spectrometry in human mast cell-derived extracellular vesicles.
- All processing and analysis steps reference record sets and fields by `@id`, following best practices for semantic data integration.
- The data can be further analyzed for biomarker discovery, annotation work, or extended by combining with additional FAIR-compliant biomedical datasets.
